# 02 — Bigram language model

The simplest possible language model: predict the next character using **only the current
character**. No context, no attention — just a lookup table of "after character A, how likely
is each character B?"

It's a weak model, but it has the exact same shape as a GPT: tokens in → scores (logits) for
the next token out → cross-entropy loss → train with gradient descent → sample to generate.
Step 3 swaps the lookup table for a transformer and keeps everything else.

## 1. Data, tokenizer, batches
Same as notebook 01, condensed. Run notebook 01 once first so `data/tinyshakespeare.txt` exists.

In [ ]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)
device = "mps" if torch.backends.mps.is_available() else "cpu"

text = Path("data/tinyshakespeare.txt").read_text()  # notebooks run from the project root
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}


def encode(s: str) -> list[int]:
    return [stoi[c] for c in s]


def decode(ids: list[int]) -> str:
    return "".join(itos[i] for i in ids)


data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

block_size = 8   # context length (the bigram model only ever uses the last character)
batch_size = 32  # independent chunks per training step


def get_batch(split: str):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)


print(f"vocab_size={vocab_size}  device={device}")

## 2. The model

The whole model is one `nn.Embedding(vocab_size, vocab_size)` — a **65 × 65 table**.
Row `i` holds the scores (*logits*) for which character comes after character `i`.

For an input batch `idx` of shape `(B, T)` (B chunks, T characters each), looking up every
token gives `logits` of shape `(B, T, C)` with `C = vocab_size`: at every position, 65 scores
for the next character.

**Loss:** `F.cross_entropy` turns the 65 scores into probabilities (softmax) and measures how
much probability the model gave to the *correct* next character. Lower is better.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size: int):
        super().__init__()
        # Row i = logits for the token that follows token i
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T) -> (B, T, C)

        if targets is None:
            return logits, None
        # cross_entropy wants (N, C) scores and (N,) targets, so flatten batch and time
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens: int):
        # idx: (B, T) tokens so far. Repeatedly predict one token and append it.
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]  # only the last position matters: (B, C)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # sample, don't just take the max
            idx = torch.cat([idx, idx_next], dim=1)  # (B, T+1)
        return idx


model = BigramLanguageModel(vocab_size).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")  # 65 * 65

## 3. Before training

A model that guesses uniformly over 65 characters has loss `-ln(1/65) ≈ 4.17`. The table
starts with random values, which is slightly *worse* than uniform (confidently wrong), so
expect a loss around 4.6. Untrained output is gibberish:

In [ ]:
import math

xb, yb = get_batch("train")
logits, loss = model(xb, yb)
print("logits shape:", tuple(logits.shape))
print(f"loss: {loss.item():.3f}   (uniform guessing: {math.log(vocab_size):.3f})")

start = torch.zeros((1, 1), dtype=torch.long, device=device)  # token 0 = '\n'
print(decode(model.generate(start, max_new_tokens=200)[0].tolist()))

## 4. Train

Each step: grab a batch → forward pass → loss → `backward()` computes gradients →
`optimizer.step()` nudges every table entry to make the correct next characters more likely.

`estimate_loss` averages the loss over many batches (one batch is noisy) on both splits, so
we can check the model isn't just memorising the training text.

In [ ]:
max_iters = 5000
eval_interval = 250
eval_iters = 100
learning_rate = 1e-2  # small model, can take big steps


@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
history = {"step": [], "train": [], "val": []}

for step in range(max_iters + 1):
    if step % eval_interval == 0:
        losses = estimate_loss()
        history["step"].append(step)
        history["train"].append(losses["train"])
        history["val"].append(losses["val"])
        if step % 1000 == 0:
            print(f"step {step:5d}: train {losses['train']:.3f}, val {losses['val']:.3f}")
    if step == max_iters:
        break

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
import matplotlib.pyplot as plt

SURFACE, INK, INK_2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
BLUE, ORANGE = "#2a78d6", "#eb6834"

fig, ax = plt.subplots(figsize=(10, 3.8), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
ax.plot(history["step"], history["train"], color=BLUE, linewidth=2, label="train")
ax.plot(history["step"], history["val"], color=ORANGE, linewidth=2, label="val")
ax.axhline(math.log(vocab_size), color=INK_2, linewidth=1, linestyle=":")
ax.text(max_iters, math.log(vocab_size), "uniform guessing", ha="right", va="bottom", fontsize=9, color=INK_2)
ax.set_xlabel("training step", color=INK_2)
ax.set_ylabel("loss", color=INK_2)
ax.set_title(f"Bigram model loss: final train {history['train'][-1]:.2f}, val {history['val'][-1]:.2f}", loc="left", color=INK)
ax.legend(frameon=False, labelcolor=INK_2, loc="center right")
ax.yaxis.grid(True, color=GRID, linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors=INK_2)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
for side in ["left", "bottom"]:
    ax.spines[side].set_color(GRID)
plt.show()

## 5. After training

Loss drops from ~4.2 to ~2.5 and plateaus — that's the best a one-character context can do.
The output now has word-like chunks, spacing and line breaks, but no real words:

In [ ]:
print(decode(model.generate(start, max_new_tokens=500)[0].tolist()))

### What did it learn?

Turn each row of the table into probabilities with softmax: row `'q'` should put almost all
its weight on `'u'`.

In [ ]:
probs = F.softmax(model.token_embedding_table.weight, dim=-1)  # (65, 65)
for ch in ["q", "\n", "."]:
    top = torch.topk(probs[stoi[ch]], 3)
    pairs = ", ".join(f"{decode([i])!r} {p:.0%}" for p, i in zip(top.values.tolist(), top.indices.tolist()))
    print(f"after {ch!r:5} -> {pairs}")

## Why it plateaus, and what's next

The model sees `t` and has no idea whether it's in "the", "it" or "what" — the other 7
characters in each chunk are ignored. The fix is to let each position **look back at the
earlier characters** and decide which ones matter. That mechanism is **self-attention**,
the core of the transformer — step 3.